# GPT-light Colab Notebook

Dieses Notebook lädt einen kleinen Chat-Datensatz, erstellt ein einfaches GPT-Modell und trainiert es auf einer kleinen Menge Daten. Es ist direkt in Google Colab ausführbar.

In [ ]:
!pip install -q datasets transformers accelerate


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


In [ ]:
raw_ds = load_dataset('HuggingFaceTB/smol-smoltalk', split='train[:500]')

def to_chat_text(example):
    parts = []
    for msg in example['messages']:
        role = msg['role'].strip().lower()
        content = msg['content'].strip()
        parts.append(f'<|{role}|>\n{content}')
    text = '\n\n'.join(parts)
    return {'text': text[:1200]}

raw_ds = raw_ds.map(to_chat_text)
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen-tokenizer')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = raw_ds['text']
print('num examples:', len(texts))
print(texts[0][:1000])
print('sample tokenized length:', len(tokenizer.encode(texts[0], add_special_tokens=False)))


In [ ]:
all_ids = []
for sample in texts:
    ids = tokenizer.encode(sample, add_special_tokens=False, truncation=True, max_length=512)
    all_ids.extend(ids)

data = torch.tensor(all_ids, dtype=torch.long)
vocab_size = tokenizer.vocab_size

print(data.shape, data.dtype)
print('first tokens:', data[:40].tolist())
print(tokenizer.decode(data[:100].tolist()))
print('vocab_size:', vocab_size)


In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print('train tokens:', len(train_data))
print('val tokens:', len(val_data))


In [ ]:
torch.manual_seed(1337)

batch_size = 16
block_size = 64

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(xb.shape, yb.shape)
print(xb.device, yb.device)


In [ ]:
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1
learning_rate = 3e-4
max_iters = 1000
eval_interval = 100
eval_iters = 50

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters')

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [ ]:
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print('final loss:', loss.item())


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=100)[0].tolist()
print(tokenizer.decode(generated))


In [ ]:
CHAT_MAX_NEW_TOKENS = 120
CHAT_TEMPERATURE = 0.7
CHAT_TOP_K = 50

def generate_reply(prompt, max_new_tokens=CHAT_MAX_NEW_TOKENS, temperature=CHAT_TEMPERATURE, top_k=CHAT_TOP_K):
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    if len(ids) == 0:
        return '[prompt could not be tokenized]'

    context = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        output = model.generate(context, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)

    generated_ids = output[0][len(ids):].tolist()
    return tokenizer.decode(generated_ids).strip()

chat_history = []
print('Commands: /reset, /exit')
print('-' * 60)
user_input = input('You: ').strip()
if user_input == '/reset':
    chat_history = []
    print('History reset.')
elif user_input == '/exit':
    print('Chat ended.')
else:
    prompt_parts = []
    for u, a in chat_history:
        prompt_parts.append(f'{u}\n{a}')
    prompt_parts.append(user_input)
    prompt = '\n'.join(prompt_parts)
    reply = generate_reply(prompt)
    chat_history.append((user_input, reply))
    print(f'Assistant: {reply}')
